# Camada BRONZE — Ingestão Bruta
Lê os CSVs e persiste como Delta Lake sem nenhuma transformação.

In [1]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark

spark = get_spark('Bronze - Ingestão')
spark.sparkContext.setLogLevel('WARN')
print('Spark iniciado:', spark.version)

Spark iniciado: 4.0.0


26/06/18 16:25:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Gera os datasets de exemplo (roda só uma vez)
import subprocess, sys
subprocess.run([sys.executable, '/opt/spark/work-dir/datasets/gerar_dados.py'], check=True)

Traceback (most recent call last):
  File "/opt/spark/work-dir/datasets/gerar_dados.py", line 6, in <module>
    from faker import Faker
ModuleNotFoundError: No module named 'faker'


CalledProcessError: Command '['/usr/bin/python3', '/opt/spark/work-dir/datasets/gerar_dados.py']' returned non-zero exit status 1.

In [3]:
DATASETS = '/opt/spark/work-dir/datasets'
BRONZE   = '/opt/spark/work-dir/warehouse/bronze'

for table in ['clientes', 'produtos', 'vendas']:
    df = (
        spark.read
        .option('header', True)
        .option('inferSchema', True)
        .csv(f'{DATASETS}/{table}.csv')
    )
    df.write.format('delta').mode('overwrite').save(f'{BRONZE}/{table}')
    print(f'[bronze/{table}] {df.count()} linhas salvas')

[bronze/clientes] 100000 linhas salvas


[bronze/produtos] 1000 linhas salvas


[bronze/vendas] 1000000 linhas salvas


In [4]:
# Inspeciona o schema de cada tabela
for table in ['clientes', 'produtos', 'vendas']:
    df = spark.read.format('delta').load(f'{BRONZE}/{table}')
    print(f'\n=== {table.upper()} ===')
    df.printSchema()
    df.show(5, truncate=False)


=== CLIENTES ===
root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- data_cadastro: date (nullable = true)
 |-- ativo: integer (nullable = true)
 |-- score_credito: double (nullable = true)



26/06/18 16:27:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+----------+--------------------+-------+------------+-------------+-----+-------------+
|cliente_id|nome                |regiao |cidade      |data_cadastro|ativo|score_credito|
+----------+--------------------+-------+------------+-------------+-----+-------------+
|1         |Brenda Alves        |Sul    |Curitiba    |2024-01-30   |1    |456.2        |
|2         |Sra. Isabelly Câmara|Sul    |Joinville   |2022-03-31   |1    |322.2        |
|3         |Cauã Rocha          |Sul    |Porto Alegre|2022-08-27   |1    |318.6        |
|4         |Dra. Aurora Pastor  |Sudeste|Santos      |2023-03-06   |1    |712.5        |
|5         |Ana Beatriz Alves   |Sul    |Porto Alegre|2023-12-16   |1    |494.5        |
+----------+--------------------+-------+------------+-------------+-----+-------------+
only showing top 5 rows

=== PRODUTOS ===
root
 |-- produto_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- preco: double (nullable = t

+----------+-----------+----------+-------+-------+-------+-------------+
|produto_id|nome       |categoria |preco  |estoque|peso_kg|fornecedor_id|
+----------+-----------+----------+-------+-------+-------+-------------+
|1         |Jaqueta 996|Roupas    |4627.57|288    |4.8    |19           |
|2         |Molho 670  |Alimentos |4907.62|232    |22.65  |13           |
|3         |Açúcar 435 |Alimentos |1323.85|181    |29.12  |46           |
|4         |Capa 179   |Automotivo|2220.72|225    |21.63  |36           |
|5         |Lençol 244 |Casa      |2806.77|56     |9.7    |50           |
+----------+-----------+----------+-------+-------+-------+-------------+
only showing top 5 rows

=== VENDAS ===
root
 |-- venda_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- produto_id: integer (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- desconto_pct: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)

+--------+----------+----------+----------+------------+---------+------+----------+-------------------+
|venda_id|cliente_id|produto_id|quantidade|desconto_pct|status   |canal |data_venda|hora_venda         |
+--------+----------+----------+----------+------------+---------+------+----------+-------------------+
|268792  |50237     |211       |1         |0           |PENDENTE |app   |2023-08-21|2026-06-18 17:14:10|
|268793  |14748     |274       |4         |20          |CONCLUIDO|online|2022-03-06|2026-06-18 19:25:14|
|268794  |95200     |848       |1         |0           |CONCLUIDO|online|2023-11-26|2026-06-18 10:25:35|
|268795  |49776     |652       |10        |20          |CANCELADO|online|2023-12-14|2026-06-18 19:08:54|
|268796  |33568     |585       |6         |20          |CONCLUIDO|online|2022-04-21|2026-06-18 11:47:34|
+--------+----------+----------+----------+------------+---------+------+----------+-------------------+
only showing top 5 rows


In [5]:
# Histórico de versões Delta
from delta.tables import DeltaTable
for table in ['clientes', 'produtos', 'vendas']:
    dt   = DeltaTable.forPath(spark, f'{BRONZE}/{table}')
    hist = dt.history(1).select('version', 'timestamp', 'operation').collect()[0]
    print(f'[{table}] versão={hist["version"]}  op={hist["operation"]}')

[clientes] versão=0  op=WRITE
[produtos] versão=0  op=WRITE
[vendas] versão=0  op=WRITE
